# GBDT（2001）

- **核心原理**：擬合殘差的梯度提升（Gradient Boosting Decision Tree）
- **變體**：XGBoost（2014）、LightGBM（2017）、CatBoost（2018）；sklearn 的 `GradientBoostingRegressor`、`HistGradientBoostingRegressor`
- **推薦資料集**：House Prices（Ames 房價；1,460 間房子、79 個特徵；目標：成交價 SalePrice）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

第十一章的 AdaBoost 是「把分錯的樣本加重權重，下一個分類器專門處理它們」，背後其實是**一次加一個函數、讓指數損失下降**（TODO 8）。Friedman (2001) 把這個想法一般化：

> 任何可微的損失 $L(y, F)$，都可以「在函數空間做梯度下降」：每一步算出損失對 $F(x_i)$ 的**負梯度**，用一棵小的**回歸樹**去擬合它，再沿著那個方向走一小步。

用平方損失時，負梯度剛好就是**殘差** $y - F$ —— 所以 GBDT 最直觀的說法是「每一棵新樹都在修正前面所有樹加起來還沒預測對的部分」。

這一章是**回歸**問題（第二章線性回歸之後第一次），重點：**回歸樹怎麼切（變異數下降）、負梯度 = 偽殘差、學習率（shrinkage）與輪數的取捨、換損失函數就能抵抗離群值、early stopping，以及現代的 histogram-based GBDT（LightGBM／XGBoost 的核心）。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：每一列是一間房子。79 個特徵混合了數值（面積 `GrLivArea`、整體品質 `OverallQual` 1–10、建造年份 `YearBuilt`、車庫可停幾台 `GarageCars`……）與類別（區域 `Neighborhood`、外牆材質、廚房品質 `KitchenQual`……）；很多欄位的缺值代表「沒有這個東西」（例如 `PoolQC` 缺值 = 沒有游泳池）。目標：成交價（美元）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$、$d$ | 訓練樣本數、特徵數（前處理後） | 約 1,168、79（以載入結果為準） |
| $y_i$ | **$\log(1 + \text{SalePrice})$**（第 1 節會說明為什麼取 log） | — |
| $F_m(x)$ | 前 $m$ 棵樹加起來的預測；$F_0$ 是常數 | — |
| $L(y, F)$ | 損失函數：平方損失 $\frac{1}{2}(y - F)^2$、絕對值損失 $\lvert y - F\rvert$、Huber | — |
| $r_{im}$ | 第 $m$ 輪第 $i$ 筆的**偽殘差**（pseudo-residual）= 負梯度 $-\partial L / \partial F(x_i)$ | $(n,)$ |
| $h_m(x)$ | 第 $m$ 棵回歸樹（擬合 $r_{\cdot m}$） | — |
| $\nu$ | 學習率（`learning_rate`，shrinkage）：$F_m = F_{m-1} + \nu h_m$ | 預設 0.1 |
| $M$ | 樹的棵數（`n_estimators`） | — |
| RMSE（log） | $\sqrt{\frac{1}{n}\sum_i (y_i - \hat y_i)^2}$，在 log 價格上算；Kaggle 這個比賽的官方指標 | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第二章 | 第一部分第 2 節：線性回歸（`01-classic-ml/02-linear-regression`） | 最小化平方誤差；梯度 $\partial \text{MSE}/\partial w$；Ridge 加 L2；殘差圖用來檢查模型錯在哪。 |
| 第二章的梯度下降 | 同上 | $w \leftarrow w - \eta \nabla_w L$；學習率太大會震盪、太小收斂慢。 |
| 第五章 | 決策樹 | 排序 + `cumsum` 一次算出所有切點的統計量；遞迴長樹；樹不需要標準化。 |
| 第十章 | Random Forest | bagging：平行地長很多棵**深**樹再**平均** → 降低變異；`max_features` 讓樹不相關；OOB。 |
| 第十一章 | AdaBoost | boosting：一個接一個加**弱**分類器 → 降低偏差；$\alpha$ 由指數損失推出；`staged_predict` 一次拿到每一輪的預測；對標錯的資料敏感（指數損失）。 |
| 第十一章 TODO 12 | 同上 | staged CV：一次 fit $T_\text{max}$ 輪，每一輪的驗證分數一次算完，再選輪數。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 平方誤差與平均**：一組數 $r_1..r_n$，讓 $\sum_i (r_i - c)^2$ 最小的常數 $c$ 是**平均** $\bar r$；最小值 = $\sum_i r_i^2 - n\bar r^2$（SSE，sum of squared errors）。例：$r = (1, 2, 6)$ → $c = 3$、SSE $= 1 + 4 + 36 - 27 = 14$（$= 4 + 1 + 9$ ✓）。

**2. 絕對誤差與中位數**：讓 $\sum_i |r_i - c|$ 最小的 $c$ 是**中位數**。例：$r = (1, 2, 60)$ → 平均 21（被 60 拉走），中位數 2。

**3. 對 $F$ 微分**（把 $F(x_i)$ 當成一個變數 $F$）：
- $L = \frac{1}{2}(y - F)^2$ → $\partial L / \partial F = -(y - F)$ → 負梯度 = $y - F$（殘差）
- $L = |y - F|$ → $\partial L / \partial F = -\text{sign}(y - F)$ → 負梯度 = $\text{sign}(y - F)$（只看方向，不看大小）

**4. 梯度下降**（第二章）：$\theta \leftarrow \theta - \eta \nabla L(\theta)$。GBDT 的「參數」就是每一筆的預測值 $F(x_1), \ldots, F(x_n)$，負梯度告訴我們每一筆該往哪裡移。但我們不能直接改 $F(x_i)$（新資料沒有 $i$），所以用一棵樹去「近似」這個方向 —— 樹會把相似的房子分在同一片葉子，給它們同樣的修正量。

**5. log 轉換**：$\log(1 + y)$ 把右偏的分布拉成接近對稱；log 的差 $\approx$ 相對誤差：$\log a - \log b = \log(a/b)$，$\log(1.1) \approx 0.095$ → log 尺度上 0.1 的誤差 ≈ 10% 的價差。

**6. 前綴和（prefix sum）**：`np.cumsum(r)` 的第 $i$ 個 = 前 $i+1$ 個的和。切點左邊的和 = 前綴和，右邊的和 = 總和 − 左邊（第五章 `cumsum` 的回歸版）。

#### 程式

**7. 回歸樹的葉子 = 平均**：`DecisionTreeRegressor` 的每個葉子預測「落在這片葉子的訓練目標的平均」。`tree.apply(X)` 回傳每一筆落在哪片葉子（節點編號）。

**8. 分組求中位數**：
```python
leaf = np.array([3, 5, 3, 5, 3]); r = np.array([1.0, 10.0, 2.0, 20.0, 9.0])
{k: np.median(r[leaf == k]) for k in np.unique(leaf)}     # {3: 2.0, 5: 15.0}
```

**9. 「等待改善」的迴圈（patience）**：記住「目前最好的分數與它在第幾輪」，每一輪看有沒有更好；連續 `patience` 輪都沒有更好就停。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ml_dl_llm.data import load_house_prices

X_train, X_test, y_train, y_test = load_house_prices()
X_train = X_train.drop(columns=["Id"], errors="ignore")
X_test = X_test.drop(columns=["Id"], errors="ignore")
y_train = pd.Series(np.asarray(y_train, dtype=float), index=X_train.index, name="SalePrice")
y_test = pd.Series(np.asarray(y_test, dtype=float), index=X_test.index, name="SalePrice")
NUM = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
CAT = [c for c in X_train.columns if c not in NUM]
train = X_train.assign(SalePrice=y_train)

print("train / test：", X_train.shape, X_test.shape, f"；數值欄 {len(NUM)}、類別欄 {len(CAT)}")
display(X_train.head(3))

In [ ]:
# 目標：成交價
# 觀察：（價格的分布對稱嗎？平均和中位數差多少？取 log 之後呢？
#        用 RMSE 評估時，一間 75 萬的豪宅錯 10% 和一間 10 萬的小房子錯 10%，在原始尺度和 log 尺度上各差多少？）
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(y_train, bins=50)
axes[0].set_title("SalePrice")
axes[1].hist(np.log1p(y_train), bins=50)
axes[1].set_title("log1p(SalePrice)")
plt.show()
print(f"平均 {y_train.mean():,.0f}、中位數 {y_train.median():,.0f}、最小 {y_train.min():,.0f}、最大 {y_train.max():,.0f}")
print("偏度（skewness）：原始", round(y_train.skew(), 2), "；log", round(np.log1p(y_train).skew(), 2))

In [ ]:
# 缺值與最相關的數值特徵
# 觀察：（缺最多的幾欄是什麼？缺值代表「不知道」還是「沒有」？與 log 價格最相關的數值特徵是哪些？）
miss = X_train.isna().mean().sort_values(ascending=False)
display(miss[miss > 0].head(12).round(3).to_frame("缺值比例").T)
corr = X_train[NUM].corrwith(np.log1p(y_train)).sort_values(key=np.abs, ascending=False)
display(corr.head(10).round(3).to_frame("r with log price").T)

In [ ]:
# 兩個最強的特徵與一個類別特徵
# 觀察：（OverallQual 與價格是直線關係嗎？GrLivArea 右下角那幾個點（面積很大、價格很低）是什麼？
#        Neighborhood 之間的價差多大？）
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.boxplot(x=X_train["OverallQual"], y=np.log1p(y_train), ax=axes[0])
axes[0].set_ylabel("log price")
axes[1].scatter(X_train["GrLivArea"], np.log1p(y_train), s=8, alpha=0.6)
axes[1].set_xlabel("GrLivArea")
axes[1].set_ylabel("log price")
odd = (X_train["GrLivArea"] > 4000) & (y_train < 300000)
axes[1].scatter(X_train.loc[odd, "GrLivArea"], np.log1p(y_train[odd]), s=60, facecolors="none", edgecolors="r", label="large & cheap")
axes[1].legend()
order = train.groupby("Neighborhood", observed=True)["SalePrice"].median().sort_values().index
sns.boxplot(x=X_train["Neighborhood"].astype(str), y=np.log1p(y_train), order=order.astype(str), ax=axes[2])
axes[2].tick_params(axis="x", rotation=90, labelsize=7)
fig.tight_layout()
plt.show()
display(train.loc[odd, ["GrLivArea", "OverallQual", "SaleCondition", "SalePrice"]])

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 為什麼本章預測 log(1 + SalePrice) 而不是 SalePrice？
#     (a) 用偏度與直方圖回答；(b) 用前置複習 5 回答：log 尺度上的 RMSE 量的是「絕對」還是「相對」誤差？
#     房價問題比較在乎哪一種？
# 答：
#
# (2) 缺值：PoolQC、MiscFeature、Alley 缺 90% 以上。這些缺值是「不知道」還是「沒有」？
#     本章的做法：類別欄補成 "Missing"、數值欄補中位數。對 GarageYrBlt（沒有車庫的房子）補中位數合理嗎？樹會受影響嗎？
# 答：
#
# (3) 「面積很大、價格很低」的那幾筆：是資料錯誤、還是特殊交易（看 SaleCondition）？
#     先預測：用平方損失訓練的模型，會被這幾筆拉走嗎？換成絕對值損失呢？（3C 的實驗 3 會驗證）
# 答：

---
## 2. 前處理與 Baseline

#### 前處理（樹模型用）

- 數值欄：缺值補中位數；類別欄：缺值補 `"Missing"`，再 `OrdinalEncoder`（第十章）。前處理後 $d = 79$ 個數值欄。
- 目標：`y = np.log1p(SalePrice)`。

#### Baseline

- **Dummy mean**：永遠預測訓練集 log 價格的平均。
- **Ridge**（第二章）：數值欄標準化、類別欄 one-hot，$\alpha = 10$。
- **單一回歸樹**（深度 3、深度 8）、**Random Forest**（第十章，300 棵）。

#### 本節名詞

- **RMSE（log）**：主要指標，越低越好。
- **5-fold CV**（回歸用 `KFold`，不是分層）；`neg_root_mean_squared_error` 是 sklearn 的「越大越好」版本，取負號換回 RMSE。

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor


def make_prep() -> ColumnTransformer:
    """樹模型用：數值補中位數、類別補 "Missing" 後 ordinal 編碼 → (n, 79) 的 DataFrame。"""
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")), ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), NUM), ("cat", cat, CAT)], verbose_feature_names_out=False).set_output(transform="pandas")


def make_linear_prep() -> ColumnTransformer:
    """線性模型用：數值補中位數 + 標準化、類別 one-hot。"""
    num = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")), ("oh", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num, NUM), ("cat", cat, CAT)])


prep = make_prep().fit(X_train)
Z_train, Z_test = prep.transform(X_train), prep.transform(X_test)
FEATURES = Z_train.columns.tolist()
# sklearn 的樹內部用 float32 存特徵；先轉一次，手寫版跟 sklearn 的門檻才會一模一樣
Xtr, Xte = Z_train.to_numpy(dtype=np.float32).astype(float), Z_test.to_numpy(dtype=np.float32).astype(float)
ytr, yte = np.log1p(y_train.to_numpy()), np.log1p(y_test.to_numpy())

cv = KFold(n_splits=5, shuffle=True, random_state=42)


def cv_rmse(model, X=X_train, y=np.log1p(y_train)) -> dict[str, str]:
    """5-fold CV 的 RMSE（log 尺度）「平均 ± 標準差」與平均 fit 秒數。"""
    s = cross_validate(model, X, y, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    return {"RMSE (log)": f"{-s['test_score'].mean():.4f} ± {s['test_score'].std():.4f}", "fit s": f"{s['fit_time'].mean():.2f}"}


baseline = pd.DataFrame(
    {
        "Dummy mean": cv_rmse(DummyRegressor()),
        "Ridge alpha=10": cv_rmse(Pipeline([("prep", make_linear_prep()), ("m", Ridge(alpha=10))])),
        "Tree max_depth=3": cv_rmse(Pipeline([("prep", make_prep()), ("m", DecisionTreeRegressor(max_depth=3, random_state=0))])),
        "Tree max_depth=8": cv_rmse(Pipeline([("prep", make_prep()), ("m", DecisionTreeRegressor(max_depth=8, random_state=0))])),
        "Random Forest 300": cv_rmse(Pipeline([("prep", make_prep()), ("m", RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1))])),
    }
).T
display(baseline)

In [ ]:
# TODO 2（問答）
#
# (1) Dummy 的 RMSE（log）大約是多少？它等於 log 價格的什麼統計量？換算成「相對誤差」大約是幾 %？
# 答：
#
# (2) 深度 3 的樹 RMSE 是多少？它最多只有幾片葉子 → 最多只能預測幾種價格？
#     GBDT 預設就是用深度 3 的樹（sklearn 的 GradientBoostingRegressor）。一棵這麼粗的樹，要怎麼「加起來」變成好模型？
# 答：
#
# (3) Ridge 和 Random Forest 誰比較好？差多少？在房價這種「很多特徵大致線性相加」的資料上，線性模型為什麼不差？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 回歸樹：用前綴和一次算出所有切點的 SSE 下降 | 3 |
| 3B | 梯度提升：負梯度（偽殘差）→ 擬合一棵樹 → 走一小步；絕對值損失的葉子要換成中位數 | 4、5、6 |
| 3C | 實驗：學習率 × 輪數、樹的深度與 subsample、損失函數與離群值 | — |

### 3A. 回歸樹

分類樹用 Gini，回歸樹用**平方誤差**：一個節點的不純度 = 節點內 $\sum_i (r_i - \bar r)^2$（SSE），葉子的預測 = 平均（前置複習 1）。一刀切成左右兩邊，SSE 下降量：

$$\Delta = \text{SSE}(\text{parent}) - \text{SSE}(L) - \text{SSE}(R) = \frac{S_L^2}{n_L} + \frac{S_R^2}{n_R} - \frac{S^2}{n}$$

其中 $S_L = \sum_{i \in L} r_i$、$S_R = S - S_L$（$\sum r_i^2$ 那一項左右相加剛好等於父節點的，消掉了）。所以**只需要左邊的和** —— 排序後用前綴和，一次算完所有切點（第五章 `cumsum` 的回歸版）。

（sklearn 的 `GradientBoostingRegressor` 預設 `criterion="friedman_mse"`：$\frac{n_L n_R}{n_L + n_R}(\bar r_L - \bar r_R)^2$。代數上跟上面的 $\Delta$ 完全相等，所以挑出來的切點一樣。）

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $x$、$r$ | 節點內樣本的一個特徵、目標（GBDT 裡是偽殘差） | $(n,)$ | `x`、`r` |
| $r_s$ | 依 $x$ 排序後的 $r$ | $(n,)$ | `rs` |
| $S_L$ | 切點 $i$ 左邊（前 $i+1$ 筆）的 $r$ 總和 | $(n-1,)$ | `S_left` |
| $S$ | 全部的和 | 純量 | `S` |
| $n_L$、$n_R$ | 左右筆數 | $(n-1,)$ | `n_left`、`n_right` |
| $\Delta$ | 每個切點的 SSE 下降 | $(n-1,)$ | `gain` |

**下標公式**（TODO 3）：$S_{L,i} = \sum_{k \le i} r_{s,k}$ → `np.cumsum(rs)[:-1]`；$\Delta_i = S_{L,i}^2/n_{L,i} + (S - S_{L,i})^2/n_{R,i} - S^2/n$（逐元素運算，沒有加總）。

#### 最小例子：TODO 3（用前綴和算 SSE 下降）

4 筆：$x = (1, 2, 3, 4)$、$r = (1, 2, 6, 7)$。

1. **正確答案**：迴圈對每個切點直接算 SSE(parent) − SSE(L) − SSE(R)。父節點 SSE $= 26$；切在 2、3 之間：左 $(1, 2)$ SSE 0.5、右 $(6, 7)$ SSE 0.5 → $\Delta = 25$（最好）。
2. **積木**：`np.cumsum(rs)`、`S − S_left`、`n_left = np.arange(1, n)`。
3. **留給你**：組出 $\Delta$。

In [ ]:
# 最小例子：TODO 3（用前綴和算 SSE 下降）
x_mini = np.array([1.0, 2.0, 3.0, 4.0])
r_mini = np.array([1.0, 2.0, 6.0, 7.0])
sse = lambda v: float(((v - v.mean()) ** 2).sum()) if len(v) else 0.0

# ---- 1. 正確答案：迴圈 ----
print("SSE(parent) =", sse(r_mini))
for i in range(3):
    L, R = r_mini[: i + 1], r_mini[i + 1 :]
    print(f"切在第 {i} 個之後：左 {L} 右 {R}  Δ = {sse(r_mini) - sse(L) - sse(R):.4f}")

# ---- 2. 積木 ----
S_left_mini = np.cumsum(r_mini)[:-1]
print("\nS_left = np.cumsum(rs)[:-1] =", S_left_mini)
print("S − S_left =", r_mini.sum() - S_left_mini, " n_left =", np.arange(1, 4), " n_right =", 4 - np.arange(1, 4))

# ---- 3. 留給你 ----
# 問題：gain = S_left²/n_left + (S − S_left)²/n_right − S²/n
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(RegressionTree()._split_gains(x_mini, r_mini)[1].round(4))

In [ ]:
from typing import Literal, Self

from numpy.typing import ArrayLike


class RNode:
    """回歸樹的節點：葉子的 value 是平均（或之後被換掉的值）。"""

    def __init__(self, value: float, n_samples: int, idx: np.ndarray) -> None:
        self.value, self.n_samples, self.idx = value, n_samples, idx
        self.feature, self.threshold, self.left, self.right = -1, np.nan, None, None

    @property
    def is_leaf(self) -> bool:
        return self.left is None


class RegressionTree:
    """CART 回歸樹（SSE 下降）。fit 後 leaves_ 是所有葉子（每片記得自己的訓練列號 idx）。"""

    def __init__(self, max_depth: int = 3, min_samples_leaf: int = 1) -> None:
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf

    def _split_gains(self, x: np.ndarray, r: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """一個特徵上所有切點 → (排序後的 x (n,), Δ (n−1,))。"""
        order = np.argsort(x, kind="stable")
        xs, rs = x[order], r[order]
        n = len(r)
        n_left = np.arange(1, n)
        n_right = n - n_left
        S = rs.sum()
        # TODO 3（程式）：前綴和算出每個切點的 SSE 下降
        #   - S_left = np.cumsum(rs)[:-1]
        #   - gain = S_left²/n_left + (S − S_left)²/n_right − S²/n
        raise NotImplementedError("TODO 3：SSE 下降")
        return xs, gain

    def _best_split(self, X: np.ndarray, r: np.ndarray) -> tuple[float, int, float]:
        best = (-np.inf, -1, np.nan)
        n = len(r)
        for j in range(X.shape[1]):
            xs, gain = self._split_gains(X[:, j], r)
            n_left = np.arange(1, n)
            valid = (xs[:-1] < xs[1:]) & (n_left >= self.min_samples_leaf) & (n - n_left >= self.min_samples_leaf)
            if not valid.any():
                continue
            i = int(np.argmax(np.where(valid, gain, -np.inf)))
            if gain[i] > best[0] + 1e-12:
                best = (float(gain[i]), j, float((xs[i] + xs[i + 1]) / 2))
        return best

    def fit(self, X: ArrayLike, r: ArrayLike) -> Self:
        self.X_, self.r_ = np.asarray(X, dtype=float), np.asarray(r, dtype=float)
        self.leaves_ = []
        self.root_ = self._grow(np.arange(len(self.r_)), 0)
        del self.X_, self.r_
        return self

    def _grow(self, idx: np.ndarray, depth: int) -> RNode:
        node = RNode(float(self.r_[idx].mean()), len(idx), idx)
        if depth < self.max_depth and len(idx) >= 2 * self.min_samples_leaf:
            gain, j, s = self._best_split(self.X_[idx], self.r_[idx])
            if j >= 0 and gain > 1e-12:
                node.feature, node.threshold = j, s
                go = self.X_[idx, j] <= s
                node.left, node.right = self._grow(idx[go], depth + 1), self._grow(idx[~go], depth + 1)
                return node
        self.leaves_.append(node)
        return node

    def apply_one(self, x: np.ndarray) -> RNode:
        node = self.root_
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node

    def predict(self, X: ArrayLike) -> np.ndarray:
        return np.array([self.apply_one(x).value for x in np.asarray(X, dtype=float)])


tree3 = RegressionTree(max_depth=3).fit(Xtr, ytr)
sk3 = DecisionTreeRegressor(max_depth=3, random_state=0).fit(Xtr, ytr)
print("✅ 深度 3：與 sklearn DecisionTreeRegressor 的預測最大差異", np.abs(tree3.predict(Xte) - sk3.predict(Xte)).max(), "（應 < 1e-10）")
print("葉子數：", len(tree3.leaves_), "；每片葉子的預測（log 價格）：", sorted(round(l.value, 3) for l in tree3.leaves_))

### 3B. 梯度提升

```
F_0 = argmin_c Σ_i L(y_i, c)        平方損失：平均；絕對值損失：中位數
for m = 1..M:
    r_i = −∂L/∂F(x_i) |_{F = F_{m−1}}                               ← TODO 4
    h_m = 在 (X, r) 上訓練的回歸樹（深度 3）
    （絕對值損失）每片葉子的值換成「葉子裡 y − F_{m−1} 的中位數」     ← TODO 6
    F_m = F_{m−1} + ν · h_m                                         ← TODO 5
```

- **平方損失**：$r_i = y_i - F_{m-1}(x_i)$ 就是殘差；樹的葉子 = 葉子裡殘差的平均 = 剛好是讓平方損失下降最多的值，不用再調。
- **絕對值損失**：$r_i = \text{sign}(y_i - F_{m-1}(x_i))$ 只有 ±1。樹用它來「分組」（決定哪些房子該一起往上、哪些往下），但葉子的值如果用 ±1 的平均，會跟真正需要的修正量完全無關 → 每片葉子改成「讓絕對值損失最小的常數」= 葉子裡**真正殘差 $y - F$ 的中位數**（前置複習 2）。這一步叫 **line search**，sklearn 也是這樣做。
- **學習率 $\nu$**：每一步只走 $\nu$ 倍（shrinkage）。$\nu$ 小，每棵樹只修一點點，需要更多棵，但通常泛化比較好。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $F$ | 目前所有樹加起來對訓練資料的預測 | $(n,)$ | `F` |
| $r$ | 偽殘差（負梯度） | $(n,)$ | `r` |
| $h_m$ | 第 $m$ 棵 `RegressionTree` | — | `tree` |
| `init_` | $F_0$（常數） | 純量 | `self.init_` |
| `trees_` | $M$ 棵樹 | 串列 | `self.trees_` |
| 葉子的 `idx` | 落在這片葉子的訓練列號 | — | `leaf.idx` |

#### 程式結構圖

```
gbm = GradientBoosting(loss="squared_error", n_estimators=100, learning_rate=0.1, max_depth=3).fit(Xtr, ytr)
├── init_ = 平均（或中位數）
├── trees_ = [RegressionTree, RegressionTree, ...]     每棵有 root_、leaves_（每片葉子有 value、idx）
└── train_loss_ (M,)：每一輪之後的訓練損失

fit：F = init_；for m → r = negative_gradient(y, F) (TODO 4) → tree.fit(X, r)
       →（absolute_error）_update_leaves(tree, y, F) (TODO 6) → F += ν · tree 在訓練資料上的預測 (TODO 5)
predict(X)        ：init_ + ν Σ_m tree.predict(X)
staged_predict(X) ：init_ + ν · cumsum_m tree.predict(X)   → (M, n)
```

訓練資料上的 `tree.predict(X)` 不用一筆一筆走：每片葉子記得自己的 `idx`，`F[leaf.idx] += ν · leaf.value` 就好。

#### 最小例子：TODO 4、5、6（一輪梯度提升）

5 筆，$y = (1, 2, 3, 4, 100)$（最後一筆是離群值）。假裝這一輪的樹把前 3 筆分到葉子 A、後 2 筆分到葉子 B。$\nu = 1$ 方便看。

1. **正確答案**：
   - 平方損失：$F_0 = 22$（平均）；$r = y - F_0 = (-21, -20, -19, -18, 78)$；葉子 A 平均 −20、B 平均 30 → $F_1 = (2, 2, 2, 52, 52)$。離群值把 B 的第 4 筆從 4 拉到 52。
   - 絕對值損失：$F_0 = 3$（中位數）；$r = \text{sign}(y - 3) = (-1, -1, 0, 1, 1)$；葉子值換成中位數：A 的 $y - F_0 = (-2, -1, 0)$ → −1；B 的 $(1, 97)$ → 49；$F_1 = (2, 2, 2, 52, 52)$ —— 2 筆的葉子，中位數 = 平均，所以這裡一樣；葉子再多一筆就不同了。
2. **積木**：`y - F`、`np.sign(y - F)`、`np.median`、`F[idx] += lr * value`。
3. **留給你**：寫進 `negative_gradient`、`_update_leaves`、`fit` 的更新那一行。

In [ ]:
# 最小例子：TODO 4、5、6（一輪梯度提升）
y5 = np.array([1.0, 2.0, 3.0, 4.0, 100.0])
leafA, leafB = np.array([0, 1, 2]), np.array([3, 4])

# ---- 1. 正確答案 ----
for loss in ["squared_error", "absolute_error"]:
    F0 = y5.mean() if loss == "squared_error" else np.median(y5)
    F = np.full(5, F0)
    r = y5 - F if loss == "squared_error" else np.sign(y5 - F)
    F1 = F.copy()
    for leaf in [leafA, leafB]:
        v = r[leaf].mean() if loss == "squared_error" else np.median((y5 - F)[leaf])
        F1[leaf] += 1.0 * v
    print(f"{loss:<15} F0 = {F0:<5} r = {r}  →  F1 = {F1}")

# ---- 2. 積木 ----
F_mini = np.full(5, 3.0)
print("\ny − F =", y5 - F_mini, "；np.sign(y − F) =", np.sign(y5 - F_mini), "← 離群值的 97 只變成 +1")
F_tmp = F_mini.copy()
F_tmp[leafB] += 1.0 * 49
print("F[idx] += lr · value →", F_tmp)

# ---- 3. 留給你 ----
# 問題：negative_gradient 依 loss 回傳哪個積木？_update_leaves 對每片葉子做什麼？fit 裡 F 怎麼更新？
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(GradientBoosting.negative_gradient("absolute_error", y5, F_mini), GradientBoosting.negative_gradient("squared_error", y5, F_mini))

In [ ]:
class GradientBoosting:
    """梯度提升回歸（平方損失或絕對值損失），基底是 RegressionTree。"""

    def __init__(self, loss: Literal["squared_error", "absolute_error"] = "squared_error", n_estimators: int = 100, learning_rate: float = 0.1, max_depth: int = 3) -> None:
        self.loss = loss
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

    @staticmethod
    def negative_gradient(loss: str, y: np.ndarray, F: np.ndarray) -> np.ndarray:
        """−∂L/∂F，(n,)。"""
        # TODO 4（程式）：偽殘差
        #   - squared_error（L = ½(y − F)²）：y − F
        #   - absolute_error（L = |y − F|）：sign(y − F)
        raise NotImplementedError("TODO 4：負梯度")

    def _loss(self, y: np.ndarray, F: np.ndarray) -> float:
        return float(np.mean((y - F) ** 2) / 2) if self.loss == "squared_error" else float(np.mean(np.abs(y - F)))

    @staticmethod
    def _update_leaves(tree: RegressionTree, y: np.ndarray, F: np.ndarray) -> None:
        """絕對值損失的 line search：每片葉子的值改成葉子裡 (y − F) 的中位數（就地修改 tree）。"""
        # TODO 6（程式）：for leaf in tree.leaves_: leaf.value = 中位數((y − F)[leaf.idx])
        raise NotImplementedError("TODO 6：葉子的中位數")

    def fit(self, X: ArrayLike, y: ArrayLike) -> Self:
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        self.init_ = float(y.mean() if self.loss == "squared_error" else np.median(y))
        F = np.full(len(y), self.init_)
        self.trees_, self.train_loss_ = [], []
        for m in range(self.n_estimators):
            r = self.negative_gradient(self.loss, y, F)
            tree = RegressionTree(max_depth=self.max_depth).fit(X, r)
            if self.loss == "absolute_error":
                self._update_leaves(tree, y, F)
            # TODO 5（程式）：F ← F + ν · h_m(X)
            #   - 不用 tree.predict(X)：每片葉子記得自己的 idx → for leaf in tree.leaves_: F[leaf.idx] += ν · leaf.value
            raise NotImplementedError("TODO 5：更新 F")
            self.trees_.append(tree)
            self.train_loss_.append(self._loss(y, F))
        return self

    def staged_predict(self, X: ArrayLike) -> np.ndarray:
        """(M, n)：第 m 列 = 前 m+1 棵樹的預測。"""
        H = np.array([t.predict(X) for t in self.trees_])
        return self.init_ + self.learning_rate * np.cumsum(H, axis=0)

    def predict(self, X: ArrayLike) -> np.ndarray:
        return self.staged_predict(X)[-1]

In [ ]:
import time
from sklearn.ensemble import GradientBoostingRegressor

rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))

# 平方損失：每一輪都跟 sklearn 比
t0 = time.perf_counter()
mine = GradientBoosting(loss="squared_error", n_estimators=100).fit(Xtr, ytr)
t_fit = time.perf_counter() - t0
sk = GradientBoostingRegressor(loss="squared_error", n_estimators=100, random_state=0).fit(Xtr, ytr)
st_mine, st_sk = mine.staged_predict(Xte), np.array(list(sk.staged_predict(Xte)))
diff = np.abs(st_mine - st_sk).max(axis=1)
first_diff = int(np.argmax(diff > 1e-8)) + 1 if (diff > 1e-8).any() else None
print(f"[squared_error] fit {t_fit:.1f} s")
print(f"  ✅ 前 10 輪與 sklearn 的 staged 預測最大差異 {diff[:10].max():.1e}（應 < 1e-8）；第一次出現差異的輪數：{first_diff}")
print(f"  test RMSE（log）：手寫 {rmse(st_mine[-1], yte):.4f}、sklearn {rmse(st_sk[-1], yte):.4f}（應相差 < 0.01）")
seeds = [GradientBoostingRegressor(n_estimators=100, random_state=s).fit(Xtr, ytr).predict(Xte) for s in range(1, 4)]
print(f"  sklearn 自己換 random_state 的預測最大差異：{max(np.abs(p - st_sk[-1]).max() for p in seeds):.4f}")

# 絕對值損失：偽殘差只有 −1／0／+1 → 很多切法的 SSE 下降一模一樣（平手），第一棵樹就可能跟 sklearn 不同。
# 所以直接檢查 TODO 6：拿 sklearn 第一棵樹的「葉子分組」，用手寫的 _update_leaves 算葉子值，應該跟 sklearn 存的一樣
sk_abs = GradientBoostingRegressor(loss="absolute_error", n_estimators=1, random_state=0).fit(Xtr, ytr)
t_sk = sk_abs.estimators_[0, 0]
leaf_id = t_sk.apply(Xtr)
fake = RegressionTree()
fake.leaves_ = [RNode(0.0, int((leaf_id == k).sum()), np.flatnonzero(leaf_id == k)) for k in np.unique(leaf_id)]
GradientBoosting._update_leaves(fake, ytr, np.full(len(ytr), np.median(ytr)))
odd_diff, even_rows = 0.0, []
for leaf, k in zip(fake.leaves_, np.unique(leaf_id)):
    sk_v = t_sk.tree_.value[k].ravel()[0]
    if leaf.n_samples % 2 == 1:
        odd_diff = max(odd_diff, abs(leaf.value - sk_v))
    else:  # 偶數筆：np.median 取中間兩個的平均；sklearn 取「較小的那個」（加權百分位數的 inverted_cdf）
        lower = np.percentile((ytr - np.median(ytr))[leaf.idx], 50, method="inverted_cdf")
        even_rows.append({"leaf n": leaf.n_samples, "mine (np.median)": leaf.value, "sklearn": sk_v, "lower median": lower})
print(f"[absolute_error] ✅ 同樣的葉子分組下，奇數筆的葉子：手寫 _update_leaves 與 sklearn 葉子值的最大差異 {odd_diff:.1e}（應 < 1e-10）")
if even_rows:
    print("  偶數筆的葉子（TODO 7 (2)(c)）：")
    display(pd.DataFrame(even_rows).round(6))
mine_abs = GradientBoosting(loss="absolute_error", n_estimators=100).fit(Xtr, ytr)
sk_abs100 = GradientBoostingRegressor(loss="absolute_error", n_estimators=100, random_state=0).fit(Xtr, ytr)
print(f"  test RMSE（log）：手寫 {rmse(mine_abs.predict(Xte), yte):.4f}、sklearn {rmse(sk_abs100.predict(Xte), yte):.4f}（應相差 < 0.02）")
seeds = [GradientBoostingRegressor(loss="absolute_error", n_estimators=100, random_state=s).fit(Xtr, ytr).predict(Xte) for s in range(1, 4)]
print(f"  sklearn 自己換 random_state 的預測最大差異：{max(np.abs(p - sk_abs100.predict(Xte)).max() for p in seeds):.4f}")

In [ ]:
# TODO 7（問答）梯度提升
#
# (1) 函數空間的梯度下降：
#     (a) 第二章的梯度下降更新的是什麼？GBDT 每一輪「更新」的是什麼？為什麼不能直接改 F(x_i)，而要用一棵樹？
#         （提示：測試資料的 x 不在訓練集裡）
#     (b) 平方損失時，樹的葉子 = 殘差的平均。用前置複習 1 說明：這剛好是讓「這片葉子裡的平方損失」下降最多的值。
#     (c) 絕對值損失時，為什麼不能直接用 sign(y − F) 的平均當葉子的值？（最小例子：sign 只有 ±1，跟需要修多少無關）
# 答：
#
# (2) 手寫 vs sklearn：
#     (a) 平方損失：前 10 輪完全一致，之後開始不同。sklearn 自己換 random_state 也會不同。這跟第五章 TODO 11 的哪個現象一樣？
#         為什麼「一輪不同」之後，後面每一輪都會跟著不同？（提示：殘差 = y − F，F 不同了）
#     (b) 絕對值損失：第一棵樹就可能不同。偽殘差 sign(y − F) 只有三種值，為什麼這會讓「平手」變得非常常見？
#         所以檢查點改成「給同樣的葉子分組，比葉子值」—— 這驗證的是 TODO 4 還是 TODO 6？
#     (c) 偶數筆的葉子：np.median 給中間兩個的平均，sklearn 給「較小的那個」。兩個都是「讓絕對值損失最小」的值嗎？
#         （提示：|r − c| 的總和在中間兩個值之間是一段平的直線）所以哪一個才「對」？
#     (d) 最後的 test RMSE 差多少？這個差距重要嗎？
# 答：
#
# (3) 跟第十一章 AdaBoost 比：AdaBoost 用「改權重」讓下一個分類器注意分錯的樣本；GBDT 用什麼讓下一棵樹注意「還沒預測好」的樣本？
#     如果把 GBDT 的損失換成指數損失、樹換成 stump，會得到什麼？
# 答：

### 3C. 實驗：學習率 × 輪數、樹的深度與 subsample、損失函數

這一節用 sklearn（`staged_predict` 一次拿到每一輪的預測）。驗證用 5-fold CV 的平均。

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| shrinkage | 學習率 $\nu < 1$：每棵樹只加 $\nu$ 倍 |
| stochastic gradient boosting | `subsample < 1`：每一輪只用隨機抽的一部分訓練資料（不放回）長樹（Friedman 2002） |
| 交互作用的階數 | 深度 $D$ 的樹，每片葉子是最多 $D$ 個條件的「且」→ 最多能表達 $D$ 個特徵之間的交互作用 |
| Huber loss | $|e| \le \delta$ 時是平方、$|e| > \delta$ 時是絕對值（線性）；兩者的折衷 |

In [ ]:
def staged_cv(make_model, X=Xtr, y=ytr, n_splits=5) -> np.ndarray:
    """回傳 (folds, M) 的驗證 RMSE：每個 fold fit 一次，用 staged_predict 拿每一輪的預測。"""
    out = []
    for tr_i, va_i in KFold(n_splits, shuffle=True, random_state=42).split(X):
        m = make_model().fit(X[tr_i], y[tr_i])
        out.append([rmse(p, y[va_i]) for p in m.staged_predict(X[va_i])])
    return np.array(out)


# 實驗 1：學習率 × 輪數（深度 3）
M_MAX = 2000
lr_curves = {}
t0 = time.perf_counter()
for lr in [1.0, 0.3, 0.1, 0.03]:
    lr_curves[lr] = staged_cv(lambda lr=lr: GradientBoostingRegressor(n_estimators=M_MAX, learning_rate=lr, random_state=0)).mean(axis=0)
print(f"實驗 1：{time.perf_counter() - t0:.0f} s")
fig, ax = plt.subplots(figsize=(8, 4))
for lr, c in lr_curves.items():
    ax.plot(np.arange(1, M_MAX + 1), c, label=f"lr = {lr}")
ax.set_xscale("log")
ax.set_ylim(0.11, 0.2)
ax.set_xlabel("number of trees (log)")
ax.set_ylabel("CV RMSE (log price)")
ax.legend()
ax.set_title("Shrinkage: learning rate vs number of trees")
plt.show()
display(pd.DataFrame({lr: {"best M": int(np.argmin(c)) + 1, "best CV RMSE": c.min(), "CV RMSE at M=2000": c[-1]} for lr, c in lr_curves.items()}).T.round(4))

In [ ]:
# 實驗 2：樹的深度與 subsample（lr = 0.05、M = 1000）
rows = []
t0 = time.perf_counter()
for depth in [1, 2, 3, 5, 8]:
    for sub in [1.0, 0.5]:
        c = staged_cv(lambda depth=depth, sub=sub: GradientBoostingRegressor(n_estimators=1000, learning_rate=0.05, max_depth=depth, subsample=sub, random_state=0)).mean(axis=0)
        rows.append({"max_depth": depth, "subsample": sub, "best M": int(np.argmin(c)) + 1, "best CV RMSE": c.min()})
print(f"實驗 2：{time.perf_counter() - t0:.0f} s")
dep_tbl = pd.DataFrame(rows)
display(dep_tbl.pivot_table(index="max_depth", columns="subsample", values=["best CV RMSE", "best M"]).round(4))

In [ ]:
# TODO 8（問答）學習率、輪數、深度
#
# (1) 實驗 1：
#     (a) lr = 1 的曲線在第幾棵樹到最低、之後怎麼變？lr = 0.03 呢？
#     (b) 「最好的 RMSE」誰比較低？「學習率小、樹多」的代價是什麼？
#     (c) 跟第十一章比：AdaBoost 的 learning_rate 也是同一個意思嗎？跟第二章梯度下降的學習率呢？
#         第二章學習率太大會「發散」；這裡 lr = 1 會發散嗎？還是只是比較快過擬合？
# 答：
#
# (2) GBDT 會過擬合嗎？用實驗 1 的曲線回答。這跟第十章 Random Forest「樹越多越好」為什麼不同？
#     （提示：RF 是「平均」，多一棵樹只是讓平均更穩；GBDT 是「加總」，每多一棵都在更貼近訓練資料的殘差）
# 答：
#
# (3) 實驗 2：
#     (a) max_depth = 1（stump，只能表達「每個特徵各自的效果相加」）和 3、5 差多少？房價有沒有明顯的交互作用？
#     (b) max_depth = 8 的最佳 M 為什麼比較小？
#     (c) subsample = 0.5 有幫助嗎？用第十章「降低樹與樹之間的相關／變異」的觀點解釋。
# 答：

In [ ]:
# 實驗 3：損失函數與離群值。在訓練資料裡人為加入 3% 的離群值（價格乘上 e³ ≈ 20 倍），測試集不動
rng_o = np.random.default_rng(0)
out_idx = rng_o.choice(len(ytr), size=int(0.03 * len(ytr)), replace=False)
y_dirty = ytr.copy()
y_dirty[out_idx] += 3.0
rows = []
for loss in ["squared_error", "absolute_error", "huber"]:
    for name, yy in [("clean", ytr), ("3% outliers", y_dirty)]:
        m = GradientBoostingRegressor(loss=loss, n_estimators=500, learning_rate=0.05, random_state=0).fit(Xtr, yy)
        rows.append({"loss": loss, "train labels": name, "test RMSE (log)": rmse(m.predict(Xte), yte), "test MAE (log)": float(np.mean(np.abs(m.predict(Xte) - yte)))})
loss_tbl = pd.DataFrame(rows).pivot_table(index="loss", columns="train labels", values=["test RMSE (log)", "test MAE (log)"])
display(loss_tbl.round(4))

# 一維的「boosting 一步一步」：只用 GrLivArea，lr = 0.3，看前幾棵樹怎麼修正殘差
j_area = FEATURES.index("GrLivArea")
x1 = Xtr[:, [j_area]]
g1 = GradientBoosting(n_estimators=50, learning_rate=0.3, max_depth=2).fit(x1, ytr)
grid = np.linspace(x1.min(), x1.max(), 400)[:, None]
st1 = g1.staged_predict(grid)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].scatter(x1, ytr, s=5, alpha=0.4)
for m in [1, 3, 10, 50]:
    axes[0].plot(grid, st1[m - 1], label=f"F_{m}")
axes[0].set_xlabel("GrLivArea")
axes[0].set_ylabel("log price")
axes[0].legend()
axes[0].set_title("Staged fit (one feature)")
F1_tr, F3_tr = g1.staged_predict(x1)[[0, 2]]
axes[1].scatter(x1, ytr - g1.init_, s=5, alpha=0.4, label="residual before tree 1")
axes[1].plot(grid, g1.trees_[0].predict(grid), "r", label="tree 1 (fits it)")
axes[1].legend()
axes[1].set_title("Round 1: residuals and the tree")
axes[2].scatter(x1, ytr - F3_tr, s=5, alpha=0.4, label="residual after 3 trees")
axes[2].plot(grid, g1.trees_[3].predict(grid), "r", label="tree 4")
axes[2].legend()
axes[2].set_title("Round 4: what is left to fix")
fig.tight_layout()
plt.show()

In [ ]:
# TODO 9（問答）損失函數與「一步一步」
#
# (1) 實驗 3：
#     (a) 加了離群值之後，三種損失的 test RMSE 各變差多少？哪一個最穩？
#     (b) 用前置複習 2、3 解釋：平方損失的負梯度 y − F 在離群值上是多大？絕對值損失的 sign(y − F) 呢？
#         這跟第十一章 AdaBoost「指數損失讓標錯的資料吃掉權重」是同一類問題嗎？
#     (c) 沒有離群值（clean）時，絕對值損失比平方損失好還是差？為什麼？Huber 想兼顧什麼？
# 答：
#
# (2) 一維的圖：
#     (a) F_1、F_3、F_10、F_50 的曲線怎麼變？為什麼是階梯？
#     (b) 中圖與右圖：第 4 棵樹在擬合的「殘差」跟第 1 棵比，大小差多少？它修正的是哪些房子？
#     (c) 最右邊（面積很大）那幾個點，F_50 有被拉過去嗎？
# 答：

---
## 4. 套件實作：`GradientBoostingRegressor`、`HistGradientBoostingRegressor`、XGBoost／LightGBM

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `GradientBoostingRegressor` | 本章手寫版的 sklearn 版本：每一輪對每個特徵排序找切點（精確，但 $n$ 大時慢） |
| `HistGradientBoostingRegressor` | 先把每個特徵**分箱**成最多 255 個 bin，找切點時只看 bin 的邊界（直方圖）→ 快很多；**原生支援缺值**（缺值自成一個方向）與**類別特徵**；內建 early stopping；用 `max_leaf_nodes` 控制樹的大小（leaf-wise，LightGBM 的做法） |
| XGBoost、LightGBM | 同樣是 GBDT，加上二階導數（牛頓法）、L1/L2 正則化、直方圖、平行化等工程最佳化；`pyproject.toml` 的 `ml` group 已安裝 |
| `feature_importances_` | 每棵樹的 SSE 下降加總（跟第五章的 impurity importance 一樣，有同樣的偏差） |

#### 程式結構圖

```
gbr = Pipeline([("prep", make_prep()), ("m", GradientBoostingRegressor(...))])
hgb = HistGradientBoostingRegressor(categorical_features="from_dtype")   ← 直接吃原始 DataFrame（類別欄是 category 型態、缺值保留）
參數名：m__learning_rate、m__n_estimators、m__max_depth、m__subsample；HGB：learning_rate、max_iter、max_leaf_nodes、min_samples_leaf、l2_regularization
```

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

X_train_hgb = X_train.copy()
X_test_hgb = X_test.copy()
for c in CAT:  # HGB 的原生類別支援需要 category 型態
    X_train_hgb[c] = X_train_hgb[c].astype("category")
    X_test_hgb[c] = pd.Categorical(X_test_hgb[c], categories=X_train_hgb[c].cat.categories)
X_train_hgb[NUM] = X_train_hgb[NUM].astype(float)  # 部分相依圖（第 7 節）不接受整數欄
X_test_hgb[NUM] = X_test_hgb[NUM].astype(float)

models = {
    "GBR default (100 trees)": (Pipeline([("prep", make_prep()), ("m", GradientBoostingRegressor(random_state=0))]), X_train),
    "GBR lr=0.05, 1000 trees, subsample=0.5": (Pipeline([("prep", make_prep()), ("m", GradientBoostingRegressor(n_estimators=1000, learning_rate=0.05, subsample=0.5, random_state=0))]), X_train),
    "HistGBR default": (HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0), X_train_hgb),
}
try:
    from xgboost import XGBRegressor

    models["XGBoost default"] = (Pipeline([("prep", make_prep()), ("m", XGBRegressor(n_jobs=1, random_state=0))]), X_train)
except ImportError:
    print("沒有安裝 xgboost，略過")
try:
    from lightgbm import LGBMRegressor

    models["LightGBM default"] = (Pipeline([("prep", make_prep()), ("m", LGBMRegressor(n_jobs=1, random_state=0, verbose=-1))]), X_train)
except ImportError:
    print("沒有安裝 lightgbm，略過")
for name, (m, Xin) in models.items():
    baseline.loc[name] = cv_rmse(m, Xin)
display(baseline)

gbr_imp = models["GBR lr=0.05, 1000 trees, subsample=0.5"][0].fit(X_train, np.log1p(y_train))
imp = pd.Series(gbr_imp["m"].feature_importances_, index=FEATURES).sort_values(ascending=False)
display(imp.head(12).round(4).to_frame("importance").T)

In [ ]:
# TODO 10（問答）
#
# (1) 讀表：
#     (a) GBR default（100 棵、lr 0.1）跟 Random Forest、Ridge 比誰好？調成 lr 0.05、1000 棵、subsample 0.5 之後呢？
#     (b) HistGBR、XGBoost、LightGBM 的預設值表現如何？fit 時間差多少？（XGBoost、LightGBM 在 CV 裡設成單執行緒，避免和 CV 的平行搶 CPU）
#     (c) 在這份只有 1,168 筆的資料上，「直方圖」的速度優勢明顯嗎？什麼樣的資料才會明顯？
# 答：
#
# (2) HistGBR 直接吃原始資料（缺值保留、類別欄不用編碼）。
#     (a) 缺值「自成一個方向」比「補中位數」好在哪裡？（提示：TODO 1 (2) 的 GarageYrBlt —— 沒有車庫的房子）
#     (b) 原生類別支援：一個節點可以把 Neighborhood 的「任意一群」類別分到左邊，ordinal 編碼只能切「字母順序的前段 / 後段」。
#         哪一個比較有彈性？有什麼風險？（類別很多時）
# 答：
#
# (3) 特徵重要度前幾名是什麼？跟 EDA 的相關係數排名一致嗎？
# 答：

---
## 5. 調參

全程只用 `X_train`。README：GBDT 用 random search 或 Optuna，**搭配 early stopping**。

| 類別 | 超參數（HistGBR） | 說明 |
| --- | --- | --- |
| 最佳化 | `learning_rate` | 越小越好但越慢；通常先固定一個小值（0.05），用 early stopping 決定輪數 |
| 容量 | `max_iter` | 輪數上限；實際輪數由 early stopping 決定 |
| 容量 | `max_leaf_nodes`、`max_depth` | 每棵樹多大 |
| 正則化 | `min_samples_leaf`、`l2_regularization` | 葉子最少幾筆；葉子值的 L2 懲罰 |

#### Early stopping

保留一小塊**驗證集**，每一輪看驗證損失；**連續 `patience` 輪都沒有比目前最好的更好**就停，回到最好的那一輪。（第十一章用 staged CV 事後挑輪數；early stopping 是「邊訓練邊挑」，不用把多餘的樹長完。）

#### 本節名詞

- **patience**（`n_iter_no_change`）：最多容忍幾輪沒進步。
- **`tol`**：「進步」至少要多少才算數。
- **`RandomizedSearchCV`**（第十章）：隨機抽參數組合。

#### 最小例子：TODO 11（手寫 early stopping）

每一輪的驗證 RMSE：`[0.30, 0.25, 0.22, 0.23, 0.21, 0.215, 0.212, 0.22, 0.23]`，patience = 3、tol = 0。

1. **正確答案**：最好的是第 5 輪（0.21）；之後第 6、7、8 輪都沒有更好 → 第 8 輪結束時停下，回報 best = 5。
2. **積木**：`best_score`、`best_iter`、`since_best` 三個變數；`enumerate(scores, start=1)`。
3. **留給你**：寫成函數。

In [ ]:
# 最小例子：TODO 11（手寫 early stopping）
val_mini = [0.30, 0.25, 0.22, 0.23, 0.21, 0.215, 0.212, 0.22, 0.23]

# ---- 1. 正確答案：一行一行追蹤 ----
best, best_it, since = np.inf, 0, 0
for it, v in enumerate(val_mini, start=1):
    if v < best:
        best, best_it, since = v, it, 0
    else:
        since += 1
    print(f"第 {it} 輪 val={v:<6} best={best:<5}（第 {best_it} 輪）連續沒進步 {since} 輪", "→ 停！" if since >= 3 else "")
    if since >= 3:
        break

# ---- 2. 積木 ----
print("\nlist(enumerate([0.3, 0.25], start=1)) =", list(enumerate([0.3, 0.25], start=1)))

# ---- 3. 留給你 ----
# 問題：early_stopping(scores, patience, tol) 回傳 (最好的輪數, 停下的輪數)。「進步」的條件是 v < best − tol。
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 (5, 8)）：
# print(early_stopping(val_mini, patience=3))

In [ ]:
def early_stopping(scores: list[float] | np.ndarray, patience: int = 10, tol: float = 0.0) -> tuple[int, int]:
    """scores：每一輪的驗證損失（越低越好）。回傳 (best_iter, stop_iter)，輪數從 1 開始；沒有停就是最後一輪。"""
    best, best_iter, since_best = np.inf, 0, 0
    # TODO 11（程式）
    #   - for it, s in enumerate(scores, start=1):
    #   - s < best − tol：更新 best、best_iter，since_best 歸零；否則 since_best += 1
    #   - since_best ≥ patience：回傳 (best_iter, it)
    #   - 迴圈跑完都沒停：回傳 (best_iter, len(scores))
    raise NotImplementedError("TODO 11：early stopping")


# 用 GBR 的 staged 驗證曲線做 early stopping（lr = 0.05，切 15% 當驗證）
from sklearn.model_selection import train_test_split

X_fit, X_val, y_fit, y_val = train_test_split(Xtr, ytr, test_size=0.15, random_state=0)
gbr_es = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05, subsample=0.5, random_state=0).fit(X_fit, y_fit)
val_curve = [rmse(p, y_val) for p in gbr_es.staged_predict(X_val)]
for patience in [10, 50, 200]:
    b, s = early_stopping(val_curve, patience=patience)
    print(f"patience = {patience:<4}：最好第 {b} 輪（val RMSE {val_curve[b - 1]:.4f}）、第 {s} 輪停下；省下 {3000 - s} 棵樹")
print("整條曲線真正的最小值：第", int(np.argmin(val_curve)) + 1, "輪")

In [ ]:
# Random search（HistGBR + 內建 early stopping）
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

hgb = HistGradientBoostingRegressor(
    categorical_features="from_dtype", max_iter=2000, learning_rate=0.05,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=0,
)
param_dist = {
    "max_leaf_nodes": randint(4, 48),
    "min_samples_leaf": randint(5, 60),
    "l2_regularization": loguniform(1e-3, 10),
    "max_features": [0.3, 0.5, 0.8, 1.0],
}
rs = RandomizedSearchCV(hgb, param_dist, n_iter=25, cv=cv, scoring="neg_root_mean_squared_error", random_state=0, n_jobs=-1)
t0 = time.perf_counter()
rs.fit(X_train_hgb, np.log1p(y_train))
print(f"random search：{time.perf_counter() - t0:.0f} s；best：{ {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rs.best_params_.items()} }；CV RMSE {-rs.best_score_:.4f}")
res = pd.DataFrame(rs.cv_results_)
res = res[[c for c in res if c.startswith("param_")] + ["mean_test_score", "std_test_score"]]
res["CV RMSE"] = -res["mean_test_score"]
display(res.sort_values("CV RMSE").drop(columns="mean_test_score").head(8).round(4))
best_hgb = rs.best_estimator_.fit(X_train_hgb, np.log1p(y_train))
print("最終模型 early stopping 停在第", best_hgb.n_iter_, "輪")

In [ ]:
# TODO 12（問答）
#
# (1) early stopping：
#     (a) patience 10、50、200 各停在哪裡？最好的那一輪一樣嗎？patience 太小會有什麼風險？（看驗證曲線的「雜訊」）
#     (b) early stopping 用掉 15% 的訓練資料當驗證集。這是什麼代價？
#     (c) 跟第十一章的 staged CV 選輪數比：哪一個比較省時間？哪一個對「輪數」的估計比較穩？
# 答：
#
# (2) random search：
#     (a) 前幾名的 max_leaf_nodes、min_samples_leaf、l2_regularization 有什麼共同點？
#     (b) 前 8 名之間的差距跟 std 比起來大嗎？
#     (c) max_features < 1 是什麼意思？（第十章的隨機特徵，用在 boosting 上）
# 答：

---
## 6. 測試集評估

測試集只評估一次。

#### 指標

- **RMSE（log）**：主要指標。
- `regression_metrics`（第二章）在**原始美元**上算 mae、rmse、r2、mape：把預測的 log 價格 `np.expm1` 換回美元。
- `log_result`／`compare`：成績寫進 `results/house_prices.csv`，依 RMSE（log）排序（越低越好）。

In [ ]:
from ml_dl_llm.metrics import regression_metrics
from ml_dl_llm.results import compare, log_result

BENCH = "house_prices"
yl_train = np.log1p(y_train)
fitted = {
    "Dummy mean": (DummyRegressor().fit(X_train, yl_train).predict(X_test), {}),
    "Ridge alpha=10 (ref)": (Pipeline([("prep", make_linear_prep()), ("m", Ridge(alpha=10))]).fit(X_train, yl_train).predict(X_test), {"alpha": 10}),
    "Tree max_depth=8 (ref)": (Pipeline([("prep", make_prep()), ("m", DecisionTreeRegressor(max_depth=8, random_state=0))]).fit(X_train, yl_train).predict(X_test), {}),
    "Random Forest 300 (ref)": (Pipeline([("prep", make_prep()), ("m", RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1))]).fit(X_train, yl_train).predict(X_test), {}),
    "GBM 100 trees (scratch)": (GradientBoosting(n_estimators=100).fit(Xtr, ytr).predict(Xte), {"n_estimators": 100, "learning_rate": 0.1}),
    "GBR default (sklearn)": (models["GBR default (100 trees)"][0].fit(X_train, yl_train).predict(X_test), {}),
    "GBR lr=0.05 1000 sub=0.5 (sklearn)": (gbr_imp.predict(X_test), {"learning_rate": 0.05, "n_estimators": 1000, "subsample": 0.5}),
    "HistGBR random-search (sklearn)": (best_hgb.predict(X_test_hgb), {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rs.best_params_.items()} | {"n_iter": int(best_hgb.n_iter_)}),
}
for name in ["XGBoost default", "LightGBM default"]:
    if name in models:
        fitted[name] = (models[name][0].fit(X_train, yl_train).predict(X_test), {})

test_scores = {}
for name, (pred_log, params) in fitted.items():
    m = {"rmse_log": round(rmse(pred_log, yte), 4), **regression_metrics(y_test, np.expm1(pred_log))}
    test_scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(test_scores).T)
compare(BENCH, "rmse_log", higher_is_better=False)

---
## 7. 診斷

#### 本節名詞

- **殘差圖**（第二章）：橫軸預測值、縱軸殘差 $y - \hat y$（log 尺度）；理想上是一團沒有形狀的雜訊。
- **最大的錯誤**：殘差絕對值最大的幾間房子；回頭看它們的特徵。
- **部分相依圖**（第十章）：看模型學到的 OverallQual、GrLivArea 與價格的關係。

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

pred_final = best_hgb.predict(X_test_hgb)
resid = yte - pred_final
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(pred_final, resid, s=10, alpha=0.6)
axes[0].axhline(0, color="r", lw=1)
axes[0].set_xlabel("predicted log price")
axes[0].set_ylabel("residual (log)")
axes[0].set_title("Residuals (HistGBR, test)")
axes[1].hist(resid, bins=40)
axes[1].set_xlabel("residual (log)")
plt.show()
print(f"殘差：|e| > 0.3（約 35% 以上的價差）的有 {int((np.abs(resid) > 0.3).sum())} 間 / {len(resid)}")
worst = pd.Series(np.abs(resid), index=X_test.index).nlargest(6).index
display(
    X_test.loc[worst, ["GrLivArea", "OverallQual", "YearBuilt", "Neighborhood", "SaleCondition"]]
    .assign(price=y_test[worst].round(0), predicted=np.expm1(pred_final[X_test.index.get_indexer(worst)]).round(0), residual=resid[X_test.index.get_indexer(worst)].round(3))
)
fig, ax = plt.subplots(figsize=(13, 3.5))
PartialDependenceDisplay.from_estimator(best_hgb, X_test_hgb, ["OverallQual", "GrLivArea", "YearBuilt"], ax=ax, n_cols=3)
plt.show()

In [ ]:
# TODO 13（問答，本章總結）
#
# (1) 錯在哪裡：
#     (a) 殘差圖有沒有明顯的形狀（例如預測值很高或很低時殘差偏一邊）？
#     (b) 錯最多的那幾間房子有什麼共同點？（SaleCondition 是不是 Abnorml／Partial？面積、品質是不是極端？）
#         這些錯誤比較像「模型不夠好」，還是「這些交易本來就不按常理」？
# 答：
#
# (2) 部分相依圖：OverallQual、GrLivArea、YearBuilt 的曲線各長什麼樣子？哪一個最接近直線？為什麼 GBDT 畫出來是階梯？
# 答：
#
# (3) 第一部分（經典機器學習）的樹模型三部曲：
#     (a) 比較表（填空）：
#
#         |                      | 決策樹（第五章） | Random Forest（第十章） | AdaBoost（第十一章） | GBDT（本章） |
#         | 基底模型             |                  |                         |                    |              |
#         | 怎麼組合             |                  |                         |                    |              |
#         | 降低偏差還是變異     |                  |                         |                    |              |
#         | 能不能平行訓練       |                  |                         |                    |              |
#         | 最重要的超參數       |                  |                         |                    |              |
#         | 容易過擬合嗎         |                  |                         |                    |              |
#         | 對離群值／雜訊       |                  |                         |                    |              |
#
#     (b) 本章 GBDT 比 Ridge 好多少（test RMSE log）？房價資料有不少「大致線性」的關係，GBDT 的優勢在哪裡？
#     (c) 「表格資料用 GBDT（XGBoost／LightGBM）」幾乎是業界的預設。用這一章的經驗說說為什麼。
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

House Prices（train / test 依載入結果；目標 log(1 + SalePrice)；CV = 5-fold；主要指標 = RMSE（log））。

| 方法 | 做法 | 關鍵性質 | CV RMSE | test RMSE |
| --- | --- | --- | --- | --- |
| Dummy mean | | | | |
| Ridge（第二章） | | | | |
| 回歸樹、Random Forest | | | | |
| GBM（手寫） | 前綴和回歸樹 + 偽殘差 | | — | |
| GBR（sklearn） | lr 0.05、1000 棵、subsample 0.5 | | | |
| HistGBR（random search + early stopping） | | | | |
| XGBoost／LightGBM 預設 | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ 函數空間梯度下降的一般形式（Friedman 2001）；分類用的 GBDT（log loss 的偽殘差 $y - p$，葉子用牛頓步）
- ☐ 二階方法：XGBoost 的目標函數 $\sum [g_i w + \frac{1}{2} h_i w^2] + \gamma T + \frac{1}{2}\lambda\lVert w\rVert^2$、葉子的最佳值 $-G/(H + \lambda)$、gain 公式
- ☐ shrinkage 為什麼有效：與 L1 正則化路徑（LARS）、forward stagewise 的關係

**工程**
- ☐ 直方圖分箱、leaf-wise vs depth-wise 長樹、GOSS、EFB（LightGBM）
- ☐ CatBoost：ordered boosting、target statistics 處理類別特徵
- ☐ 單調限制（monotonic constraints）、交互作用限制
- ☐ Optuna 調參（README 建議參數多時使用）

**特徵與評估**
- ☐ 房價的特徵工程：總面積、屋齡、品質 × 面積；log 轉換右偏的特徵（對線性模型）
- ☐ 移除 GrLivArea > 4000 的離群值對各模型的影響
- ☐ SHAP 值解釋 GBDT
- ☐ 模型堆疊（stacking）：Ridge + GBDT